In [ ]:
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession

builder = SparkSession.builder \
    .appName("DeltaExample") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

spark = configure_spark_with_delta_pip(builder).getOrCreate()

26/05/07 11:41:40 WARN Utils: Your hostname, Maicou resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/05/07 11:41:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/maicou/projetos/trabalho-spark/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/maicou/.ivy2/cache
The jars for the packages stored in: /home/maicou/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-b0dbd331-40be-41de-a55e-1e57e3573f34;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.1.0 in central
	found io.delta#delta-storage;3.1.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 213ms :: artifacts dl 9ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.1.0 from central in [default]
	io.delta#delta-storage;3.1.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0 

In [2]:
data = [
    (1, "Notebook", 3000),
    (2, "Mouse", 100),
    (3, "Teclado", 200)
]

df = spark.createDataFrame(data, ["id", "produto", "preco"])

In [3]:
df.write.format("delta").mode("overwrite").save("/tmp/vendas")

26/05/07 11:41:50 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [4]:
df = spark.read.format("delta").load("/tmp/vendas")
df.show()

+---+--------+-----+
| id| produto|preco|
+---+--------+-----+
|  1|Notebook| 3000|
|  3| Teclado|  200|
|  2|   Mouse|  100|
+---+--------+-----+



In [5]:
novo = spark.createDataFrame([(4, "Monitor", 1200)], ["id", "produto", "preco"])

novo.write.format("delta").mode("append").save("/tmp/vendas")

In [6]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "/tmp/vendas")

deltaTable.update(
    condition="id = 2",
    set={"preco": "150"}
)

In [7]:
spark.read.format("delta").load("/tmp/vendas").show()

+---+--------+-----+
| id| produto|preco|
+---+--------+-----+
|  1|Notebook| 3000|
|  4| Monitor| 1200|
|  3| Teclado|  200|
|  2|   Mouse|  150|
+---+--------+-----+



In [8]:
from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, "/tmp/vendas")

In [9]:
deltaTable.delete("id = 3")

In [10]:
spark.read.format("delta").load("/tmp/vendas").show()

+---+--------+-----+
| id| produto|preco|
+---+--------+-----+
|  1|Notebook| 3000|
|  4| Monitor| 1200|
|  2|   Mouse|  150|
+---+--------+-----+

